# 모듈 5. Human-in-the-loop 승인

이 노트북은 LangGraph `interrupt`를 사용해 위험한 Tool 실행 전에 사람의 승인을 받는 흐름을 만듭니다.

## 학습 목표

- 즉시 실행 Agent와 승인 기반 Agent의 차이를 설명한다.
- 위험 작업을 실행 계획과 실행 단계로 분리한다.
- `interrupt`와 `Command(resume=...)`로 그래프를 멈췄다가 재개한다.
- 승인/거절 결과에 따라 실행 또는 중단한다.

## 왜 LangGraph 구조로 배워야 할까?

Human-in-the-loop는 단순한 `input()` 호출이 아닙니다. 운영 Agent에서는 그래프 실행을 멈추고, 승인 요청을 외부 UI나 관리자 화면으로 보내고, 나중에 승인 값을 받아 이어서 실행해야 합니다.

LangGraph의 `interrupt`는 바로 이 문제를 해결합니다.

In [ ]:
from typing import TypedDict

from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


@tool
def refund_payment(user_id: str, amount: int, reason: str) -> str:
    """사용자에게 결제 금액을 환불한다."""
    return f"{user_id}에게 {amount:,}원 환불 완료. 사유: {reason}"


refund_request = {
    "user_id": "user-123",
    "amount": 300_000,
    "reason": "프리미엄 플랜 오결제",
}

## A안. 위험 Tool 즉시 실행

환불, 이메일 발송, 계정 삭제처럼 외부 세계를 바꾸는 Tool은 즉시 실행하면 되돌리기 어렵습니다.

In [ ]:
print(refund_payment.invoke(refund_request))

## B안. `interrupt`로 승인받고 이어서 실행하기

그래프는 먼저 승인 요청을 만들고 멈춥니다. 사람이 승인하면 `Command(resume=...)`로 이어서 실행합니다.

In [ ]:
class RefundState(TypedDict, total=False):
    user_id: str
    amount: int
    reason: str
    approval_request: dict
    approved: bool
    result: str


def plan_refund_node(state: RefundState) -> RefundState:
    return {
        "approval_request": {
            "action": "refund_payment",
            "user_id": state["user_id"],
            "amount": state["amount"],
            "reason": state["reason"],
            "risk_level": "high",
        }
    }


def approval_node(state: RefundState) -> RefundState:
    decision = interrupt(state["approval_request"])
    if isinstance(decision, dict):
        approved = bool(decision.get("approved", False))
    else:
        approved = bool(decision)
    return {"approved": approved}


def execute_refund_node(state: RefundState) -> RefundState:
    if not state.get("approved"):
        return {"result": "중단: 사람이 환불 요청을 승인하지 않았습니다."}
    result = refund_payment.invoke(
        {
            "user_id": state["user_id"],
            "amount": state["amount"],
            "reason": state["reason"],
        }
    )
    return {"result": result}


builder = StateGraph(RefundState)
builder.add_node("plan_refund", plan_refund_node)
builder.add_node("approval", approval_node)
builder.add_node("execute_refund", execute_refund_node)
builder.add_edge(START, "plan_refund")
builder.add_edge("plan_refund", "approval")
builder.add_edge("approval", "execute_refund")
builder.add_edge("execute_refund", END)

refund_graph = builder.compile(checkpointer=MemorySaver())

In [ ]:
config = {"configurable": {"thread_id": "refund-demo-001"}}

first_result = refund_graph.invoke(refund_request, config=config)
print("첫 실행 결과:")
print(first_result)

approved_result = refund_graph.invoke(
    Command(resume={"approved": True}),
    config=config,
)
print("\n승인 후 결과:")
print(approved_result["result"])

## 거절 케이스도 확인하기

같은 그래프를 새 thread로 실행하면 거절 흐름도 테스트할 수 있습니다.

In [ ]:
reject_config = {"configurable": {"thread_id": "refund-demo-002"}}

refund_graph.invoke(refund_request, config=reject_config)
rejected_result = refund_graph.invoke(
    Command(resume={"approved": False}),
    config=reject_config,
)

print(rejected_result["result"])

## 정리

- Human-in-the-loop는 위험 Tool만 선별 적용해야 합니다.
- LangGraph `interrupt`는 그래프 실행을 멈추고 외부 승인 값을 기다릴 수 있게 합니다.
- 승인 요청에는 action, payload, risk level, requester, created_at 같은 감사 정보를 포함하는 것이 좋습니다.

## 실습 과제

1. 10만 원 이하는 자동 승인, 10만 원 초과는 `interrupt`로 보내도록 분기해보세요.
2. 승인 요청에 `approver`, `approved_at`을 추가하고 결과 상태에 남겨보세요.